# Paper-aligned MTI/MTR Training (3D)

This notebook validates data/config paths, confirms independent output volumes, and builds GPU training commands aligned with the repository implementation and Gao (2024).

In [ ]:
import sys, importlib
print('Python:', sys.executable)
for name in ['torch','lightning','torchmetrics','numpy','matplotlib']:
    try:
        m = importlib.import_module(name)
        print(f'{name}: {getattr(m, "__version__", "unknown")}')
    except Exception as e:
        print(f'{name}: missing ({e})')

try:
    import torch
    print('CUDA available:', torch.cuda.is_available())
    print('CUDA devices:', torch.cuda.device_count())
    if torch.cuda.is_available():
        print('GPU 0:', torch.cuda.get_device_name(0))
except Exception as e:
    print('Torch check failed:', e)

In [ ]:
from pathlib import Path
WORKSPACE = Path('/home/roderickperez/DataScienceProjects/multiTaskLearningSeismic')
PATHS = {
    'src_mti': WORKSPACE / 'src/main3_infer.py',
    'src_mtr': WORKSPACE / 'src/main3_refine.py',
    'pyproject': WORKSPACE / 'pyproject.toml',
    'setup_guide': WORKSPACE / 'SETUP_GUIDE.md',
    'workflow_guide': WORKSPACE / 'MODEL_WORKFLOW_GUIDE.md',
    'paper_txt': WORKSPACE / 'pdf_content.txt',
    'paper_pdf': WORKSPACE / 'reference/Iterative multitask learning and inference from seismic images.pdf',
    'train_dataset3': WORKSPACE / 'train/dataset3',
    'train_dataset2': WORKSPACE / 'train/dataset2',
    'test_opunake2': WORKSPACE / 'test/opunake2',
    'test_opunake3': WORKSPACE / 'test/opunake3',
    'field_sgy': WORKSPACE / 'data/1_Original_Seismics.sgy'
}
for k, p in PATHS.items():
    print(f'{k:14} -> {p} | exists={p.exists()}')

In [ ]:
import re, numpy as np
DIR_DATA_TRAIN = WORKSPACE / 'train/dataset3/data_train'
DIR_TARGET_TRAIN = WORKSPACE / 'train/dataset3/target_train'
DIR_DATA_VALID = WORKSPACE / 'train/dataset3/data_valid'
DIR_TARGET_VALID = WORKSPACE / 'train/dataset3/target_valid'

def numeric_ids(folder: Path):
    out = []
    for f in sorted(folder.glob('*.bin')):
        m = re.match(r'^(\d+)\.bin$', f.name)
        if m:
            out.append(m.group(1))
    return out

def audit(data_dir: Path, target_dir: Path, name: str):
    ids = numeric_ids(data_dir)
    print(f'\n[{name}] ids={ids}')
    data_sfx = ['', '_rgt', '_dhr', '_fsem', '_fdip', '_fstrike']
    tgt_sfx = ['_rgt', '_dhr', '_fsem', '_fdip', '_fstrike']
    complete = True
    for sid in ids:
        md = [f'{sid}{s}.bin' for s in data_sfx if not (data_dir / f'{sid}{s}.bin').exists()]
        mt = [f'{sid}{s}.bin' for s in tgt_sfx if not (target_dir / f'{sid}{s}.bin').exists()]
        if md or mt:
            complete = False
        print(f'  id {sid}: missing_data={md if md else []}, missing_target={mt if mt else []}')
    return ids, complete

train_ids, train_ok = audit(DIR_DATA_TRAIN, DIR_TARGET_TRAIN, 'train')
valid_ids, valid_ok = audit(DIR_DATA_VALID, DIR_TARGET_VALID, 'valid')
print('\nSummary:')
print('ntrain:', len(train_ids), 'complete:', train_ok)
print('nvalid:', len(valid_ids), 'complete:', valid_ok)

f0 = DIR_DATA_TRAIN / '0.bin'
if f0.exists():
    n = np.fromfile(f0, dtype=np.float32).size
    side = int(round(n ** (1.0/3.0)))
    print('cube check:', f0.name, 'elements=', n, 'inferred side=', side)

## Independent output volumes

`src/main3_infer.py` and `src/main3_refine.py` define separate heads and write separate volumes for `rgt`, `dhr`, `fsem`, `fdip`, `fstrike`.

In [ ]:
output_keys = ['rgt', 'dhr', 'fsem', 'fdip', 'fstrike']
suffix = {'rgt':'.rgt','dhr':'.dhr','fsem':'.fsem','fdip':'.fdip','fstrike':'.fstrike'}
print('Independent output volumes expected:')
for k in output_keys:
    print(f'  {k} -> {suffix[k]}')

## Paper alignment notes

Paper-scale 3D training uses much larger synthetic sets (2000 train / 100 valid) with LR=0.5e-4 and batch=8.
Your local `train/dataset3` is a minimal demo (1/1), so poor field-volume quality is expected.

In [ ]:
DEMO = {
    'n1': 64, 'n2': 64, 'n3': 64,
    'ntrain': max(1, len(train_ids)),
    'nvalid': max(1, len(valid_ids)),
    'batch_train': 1, 'batch_valid': 1,
    'epochs': 100, 'lr': 0.5e-4, 'gpus_per_node': 1
}
PAPER_TEMPLATE = {
    'n1': 256, 'n2': 128, 'n3': 128,
    'ntrain': 2000, 'nvalid': 100,
    'batch_train': 8, 'batch_valid': 8,
    'epochs': 100, 'lr': 0.5e-4, 'gpus_per_node': 8
}
print('DEMO config:', DEMO)
print('PAPER template:', PAPER_TEMPLATE)

In [ ]:
import subprocess
OUT_MTI = WORKSPACE / 'result3_infer_new'
OUT_MTR = WORKSPACE / 'result3_refine_new'
OUT_MTI.mkdir(parents=True, exist_ok=True)
OUT_MTR.mkdir(parents=True, exist_ok=True)
cfg = DEMO.copy()

MTI_CMD = [
    'uv','run','python','src/main3_infer.py',
    f'--n1={cfg["n1"]}', f'--n2={cfg["n2"]}', f'--n3={cfg["n3"]}',
    f'--ntrain={cfg["ntrain"]}', f'--nvalid={cfg["nvalid"]}',
    f'--batch_train={cfg["batch_train"]}', f'--batch_valid={cfg["batch_valid"]}',
    f'--epochs={cfg["epochs"]}', f'--lr={cfg["lr"]}',
    f'--gpus_per_node={cfg["gpus_per_node"]}',
    f'--dir_data_train={DIR_DATA_TRAIN}',
    f'--dir_target_train={DIR_TARGET_TRAIN}',
    f'--dir_data_valid={DIR_DATA_VALID}',
    f'--dir_target_valid={DIR_TARGET_VALID}',
    f'--dir_output={OUT_MTI}',
    '--rgt=y','--dhr=y','--fault=y'
]

# Strict paper logic: MTR should use MTI-predicted multimodal inputs.
# Current local demo points to dataset3 multimodal files already present.
MTR_CMD = [
    'uv','run','python','src/main3_refine.py',
    f'--n1={cfg["n1"]}', f'--n2={cfg["n2"]}', f'--n3={cfg["n3"]}',
    f'--ntrain={cfg["ntrain"]}', f'--nvalid={cfg["nvalid"]}',
    f'--batch_train={cfg["batch_train"]}', f'--batch_valid={cfg["batch_valid"]}',
    f'--epochs={cfg["epochs"]}', f'--lr={cfg["lr"]}',
    f'--gpus_per_node={cfg["gpus_per_node"]}',
    f'--dir_data_train={DIR_DATA_TRAIN}',
    f'--dir_target_train={DIR_TARGET_TRAIN}',
    f'--dir_data_valid={DIR_DATA_VALID}',
    f'--dir_target_valid={DIR_TARGET_VALID}',
    f'--dir_output={OUT_MTR}',
    '--rgt=y','--dhr=y','--fault=y'
]

print('MTI command:')
print(' '.join(map(str, MTI_CMD)))
print('\nMTR command:')
print(' '.join(map(str, MTR_CMD)))

RUN_MTI = False
RUN_MTR = False
if RUN_MTI:
    r = subprocess.run(MTI_CMD, cwd=str(WORKSPACE), check=False)
    print('MTI exit code:', r.returncode)
if RUN_MTR:
    r = subprocess.run(MTR_CMD, cwd=str(WORKSPACE), check=False)
    print('MTR exit code:', r.returncode)
if not RUN_MTI and not RUN_MTR:
    print('Training disabled. Set RUN_MTI / RUN_MTR to True.')

In [ ]:
def show_artifacts(folder: Path, name: str):
    ckpts = sorted(folder.glob('*.ckpt'))
    print(f'\n[{name}] {folder}')
    print('ckpt count:', len(ckpts))
    for p in ckpts[-8:]:
        print('  ', p.name)
    print('has last.ckpt:', (folder / 'last.ckpt').exists())
    print('has loss_plot.png:', (folder / 'loss_plot.png').exists())

show_artifacts(OUT_MTI, 'MTI')
show_artifacts(OUT_MTR, 'MTR')